In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pymysql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

# 设置绘图风格与中文字体
plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="white", font='SimHei')

def get_db_connection():
    """建立 StarRocks 数据库连接"""
    return pymysql.connect(
        host='192.168.144.101',
        port=9030,
        user='hadoop',
        password='hadoop',
        charset='utf8'
    )

def fetch_cities():
    """获取 DWD 表中有数据的城市列表"""
    conn = get_db_connection()
    query = """
    SELECT DISTINCT city 
    FROM iceberg_catalog.ershoufang.dwd_ershoufang_fact 
    WHERE city IS NOT NULL AND city != ''
    ORDER BY city
    """
    try:
        df = pd.read_sql(query, conn)
        return df['city'].tolist()
    finally:
        conn.close()

def fetch_house_type_data(city_name):
    """
    获取指定城市的户型聚合数据。
    在 SQL 层完成多字段拼接 (室、厅、卫) 和全局聚合排序，提取 TOP 15。
    """
    conn = get_db_connection()
    query = f"""
    SELECT 
        concat(room_num, '室', hall_num, '厅', bathroom_num, '卫') as room_type,
        COUNT(house_id) as house_count
    FROM iceberg_catalog.ershoufang.dwd_ershoufang_fact 
    WHERE city = '{city_name}' 
      AND room_num IS NOT NULL 
      AND hall_num IS NOT NULL
      AND bathroom_num IS NOT NULL
    GROUP BY room_num, hall_num, bathroom_num
    ORDER BY house_count DESC
    LIMIT 15
    """
    try:
        df = pd.read_sql(query, conn)
        return df
    finally:
        conn.close()

In [ ]:
def plot_gradient_lollipop(city):
    """绘制渐变色棒棒糖图"""
    df = fetch_house_type_data(city)
    
    if df.empty:
        plt.figure(figsize=(10, 6))
        plt.text(0.5, 0.5, f"{city} 暂无有效的户型数据", ha='center', va='center', fontsize=15)
        plt.axis('off')
        plt.show()
        return

    # --- 1. 颜色渐变映射配置 ---
    norm = mcolors.Normalize(vmin=df['house_count'].min(), vmax=df['house_count'].max())
    cmap = cm.get_cmap('plasma')
    colors = cmap(norm(df['house_count']))

    # --- 2. 开始绘图 ---
    fig, ax = plt.subplots(figsize=(14, 7))
    
    x_pos = np.arange(len(df['room_type']))
    counts = df['house_count']
    ax.vlines(x=x_pos, ymin=0, ymax=counts, color=colors, linewidth=3.5, alpha=0.7)
    sc = ax.scatter(
        x_pos, counts, 
        s=250,                   
        c=counts, cmap='plasma', 
        zorder=3,                
        edgecolor='white', linewidth=1.5
    )

    # --- 3. 添加数值与修饰 ---
    # 在圆点上方添加具体的数值标签
    y_max = counts.max()
    for i, count in enumerate(counts):
        ax.text(
            x_pos[i], count + (y_max * 0.03), 
            f'{count:,}', 
            ha='center', va='bottom', fontsize=11, fontweight='bold', color='#333333'
        )

    plt.title(f'[{city}] 热门二手房户型排行榜 (TOP 15)', fontsize=16, pad=20)
    plt.xlabel('户型', fontsize=12, labelpad=10)
    plt.ylabel('房源挂牌数量 (套)', fontsize=12)
    
    # 设置 X 轴标签
    ax.set_xticks(x_pos)
    ax.set_xticklabels(df['room_type'], fontsize=12, rotation=45, ha='right')
    sns.despine(left=True, top=True, right=True)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    ax.set_ylim(0, y_max * 1.15)

    plt.tight_layout()
    plt.show()

In [3]:
def render_interactive_dashboard():
    """构建交互式下拉组件"""
    cities = fetch_cities()
    if not cities:
        print("未获取到城市列表，请检查数据。")
        return
    
    default_city = '北京' if '北京' in cities else cities[0]
    
    city_dropdown = widgets.Dropdown(
        options=cities,
        value=default_city,
        description='📍 分析城市:',
        layout={'width': '250px'}
    )
    
    plot_output = widgets.Output()

    def on_city_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with plot_output:
                clear_output(wait=True)
                plot_gradient_lollipop(change['new'])

    city_dropdown.observe(on_city_change)

    display(city_dropdown)
    display(plot_output)
    
    with plot_output:
        plot_gradient_lollipop(default_city)

In [4]:
if __name__ == "__main__":
    render_interactive_dashboard()

Dropdown(description='📍 分析城市:', index=4, layout=Layout(width='250px'), options=('上海', '东莞', '中山', '佛山', '北京', …

Output()